# Quickstart: your first forecast

Generate a small advection–diffusion dataset with AutoSim, train a deterministic U-Net with Lightning, and plot a forecast. Everything uses AutoCast's Python API; no CLI commands or Hydra configuration are needed.

After [installation](../installation.md), run `uv sync --extra dev --extra docs` from the repository root and select that environment's Python kernel. Run this notebook from its own directory; data is regenerated under `outputs/quickstart/data`.

In [ ]:
import lightning as L
import matplotlib.pyplot as plt
import torch

from autocast.data.utils import get_autosim_datamodule
from autocast.decoders.channels_last import ChannelsLast
from autocast.encoders.permute_concat import PermuteConcat
from autocast.models.encoder_decoder import EncoderDecoder
from autocast.models.encoder_processor_decoder import EncoderProcessorDecoder
from autocast.processors.unet import AzulaUNetProcessor
from autocast.utils import get_optimizer_config
from autocast.utils.plots import plot_spatiotemporal_snapshots

torch.set_num_threads(2)
_ = L.seed_everything(7)

## Generate data and inspect a batch

AutoCast's AutoSim helper generates six training trajectories and two each for validation and testing. It handles windowing, batching and field normalization using training-set statistics. Predict two frames from the preceding two.

In [ ]:
datamodule = get_autosim_datamodule(
    simulation_name="advection_diffusion",
    simulator_kwargs={"n": 16, "T": 1.25, "dt": 0.25},
    n_train=6,
    n_valid=2,
    n_test=2,
    n_steps_input=2,
    n_steps_output=2,
    stride=2,
    batch_size=4,
    num_workers=0,
    data_path="outputs/quickstart/data",
    overwrite=True,
    seed=7,
)

batch = next(iter(datamodule.train_dataloader()))
batch.input_fields.shape, batch.output_fields.shape

The shape is `(batch, time, height, width, channels)`. Each trajectory also carries its viscosity and advection parameters in `batch.constant_scalars`.

## Compose the model

The encoder and decoder here only rearrange tensors: they fold time into channels for the U-Net and restore the output layout. There is no learned compression. The processor predicts the next fields directly, conditioned on the simulation parameters.

In [ ]:
encoder_decoder = EncoderDecoder(
    encoder=PermuteConcat(in_channels=1, n_steps_input=2),
    decoder=ChannelsLast(output_channels=1, time_steps=2),
)
processor = AzulaUNetProcessor(
    in_channels=2,  # two input frames, one field each
    out_channels=2,
    hid_channels=(8, 16),
    hid_blocks=(1, 1),
    global_cond_channels=2,
    include_global_cond=True,
)
model = EncoderProcessorDecoder(
    encoder_decoder=encoder_decoder,
    processor=processor,
    loss_func=torch.nn.MSELoss(),
    optimizer_config=get_optimizer_config(learning_rate=3e-3),
    norm=datamodule.train_dataset.norm,
)

print(f"{sum(parameter.numel() for parameter in model.parameters()):,} parameters")

## Train with Lightning

This small CPU run demonstrates the API, not benchmark accuracy. Increase the number of trajectories and epochs for a useful forecasting model.

In [ ]:
trainer = L.Trainer(
    max_epochs=20,
    accelerator="cpu",
    logger=False,
    enable_checkpointing=False,
    enable_progress_bar=False,
    enable_model_summary=False,
)
trainer.fit(model, datamodule=datamodule)

## Forecast a held-out trajectory

Roll out two windows, feeding each prediction back as input. `model.rollout` returns predictions and reference fields in physical units because we supplied the training normalizer.

In [ ]:
rollout_batch = next(iter(datamodule.rollout_test_dataloader()))

model.eval()
with torch.no_grad():
    prediction, truth = model.rollout(
        rollout_batch,
        stride=2,
        max_rollout_steps=2,
        free_running_only=True,
    )

prediction.shape

Change `trajectory` below to inspect either test trajectory without retraining.

In [ ]:
trajectory = 0
figure = plot_spatiotemporal_snapshots(
    true=truth,
    pred=prediction,
    batch_idx=trajectory,
    timesteps=range(4),
    title="Held-out advection-diffusion forecast",
)
plt.show()

For a complete experiment with saved configurations, checkpoints and evaluation outputs, follow the [CLI walkthrough](../walkthrough/index.md). To explore compression, generative models and uncertainty, see the [focused notebooks](index.md).